In [3]:
from pathlib import Path
import re
import datetime

# ===== Experiment setup =====
MODEL = "llama3.3-70b"
TEMPERATURE = 0
CONFIG_NAME = "C-Full"

scenario_name = "Aircraft maintenance routing system"


def slugify(s: str) -> str:
    s = s.strip().lower()
    
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return re.sub(r"_+", "_", s).strip("_")

# ===== Create log file =====
logs_dir = Path("logs")
logs_dir.mkdir(exist_ok=True)

stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
txt_path = logs_dir / f"aircraft_maintenance_routing_system__c_full__20260218-150405.txt"
#txt_path = logs_dir / f"{slugify(scenario_name)}__{slugify(CONFIG_NAME)}__{stamp}.txt"

"""with txt_path.open("w", encoding="utf-8") as f:
    f.write("EXPERIMENT LOG\n")
    f.write("=" * 70 + "\n")
    f.write(f"SCENARIO: {scenario_name}\n")

    f.write(f"CONFIG: {CONFIG_NAME}\n")
    f.write(f"MODEL: {MODEL}\n")
    f.write(f"TEMPERATURE: {TEMPERATURE}\n")
    f.write(f"CREATED_AT: {stamp}\n")
    f.write("=" * 70 + "\n\n")"""

print("Log file created:", txt_path)


Log file created: logs/aircraft_maintenance_routing_system__c_full__20260218-150405.txt


In [4]:
def dict_to_bullets(d, indent: int = 4) -> str:
   
    pad = " " * indent
    lines = []

    if d is None or d == {}:
        return f"{pad}- (none)"

    def walk(node, level: int, show_key: bool = True):
        pad2 = " " * level

        if isinstance(node, dict):
            if not node:
                lines.append(f"{pad2}- (none)")
                return
            for k, v in node.items():
                # Skip printing the first-level key labels
                if show_key:
                    key = str(k) if k is not None and str(k).strip() else "(unnamed)"
                    lines.append(f"{pad2}- {key}:")
                    walk(v, level + 2, show_key=True)
                else:
                    # First level: don't print key, just descend into its value
                    walk(v, level, show_key=True)

        elif isinstance(node, (list, tuple, set)):
            node = list(node)
            if not node:
                lines.append(f"{pad2}- (none)")
            else:
                for item in node:
                    if isinstance(item, (dict, list, tuple, set)):
                        lines.append(f"{pad2}-")
                        walk(item, level + 2, show_key=True)
                    else:
                        lines.append(f"{pad2}- {item}")

        else:
            val = str(node).strip()
            lines.append(f"{pad2}- {val if val else '(empty)'}")

    # Start at root with show_key=False so first-level keys are hidden
    walk(d, indent, show_key=False)
    return "\n".join(lines)


In [5]:
iteration_id= 5

In [6]:
metamodel_spec = """SCM metamodel:

Types  and their fields:

- Flow:
  attributes: name, type, source, destination, frequency, volume(NUMBER)

- Resource:
  attributes: name, type, quantity(NUMBER), cost(NUMBER)
  relations:
    - isUsed[*] -> Process

- Process:
  attributes: name, id, duration, output
  relations:
    - measures[*] -> Metric
    - uses[*] -> Resource
    - produces[*] -> Resource (containment)
    - involves[*] -> Relationship
    - have[*] -> Constraint

- Entity:
  attributes: name, id, location, role
  relations:
    - measures[*] -> Metric (containment)
    - collaborates[*] -> Relationship
    - performs -> Process

- Metric:
  attributes: name, type, value(NUMBER), target(NUMBER)
  relations:
    - evaluates_ -> Relationship
    - evaluates -> Process

- Constraint:
  attributes: name, type, details
  relations:
    - applies_to[*] (required) -> Process

- Relationship:
  attributes: name, type, terms, startDate, endDate

- Ecosystem:
  attributes: name
  relations (all containment, many):
    - entitys[*] -> Entity
    - resources[*] -> Resource
    - relationships[*] -> Relationship
    - constraints[*] -> Constraint
    - metrics[*] -> Metric
    - flows[*] -> Flow
    - processs[*] -> Process

Subtypes of Flow:
- MaterialFlow extends Flow:
  relations:
    - targetProcess (required) -> Process
    - sourceProcess[*] -> Process
- InformationFlow extends Flow:
  relations:
    - targetProcess (required) -> Process
    - sourceProcess[*] -> Process

Output rules:
- Only propose additions that use the concepts listed above.
- When proposing a relation, use the exact relation name from the metamodel (e.g., performs, uses, measures, collaborates, applies_to, flows, entitys, etc.).
- no need to generate attributes of the concepts, just the concept names and their relationships.

"""
ontology_annotation = """SCM ontology mapping (use exactly these):
- Actor -> Entity
- Item -> Resource
- Interaction -> relationship, InformationFlow, MaterialFlow
- Property -> Metric
"""


In [ ]:
from openai import OpenAI

client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="local-dummy")

def generate_completion(
    scenario_name: str,
    elements: dict,
    model: str,
     dsl: str ,
    temperature: float = 0.0,
    n_additions: int = 3,
   
    ontology_annotation: str = "",
    metamodel_spec: str = "",
) -> tuple[str, str]:
    

    # Build partial model text from elements
    elems_block =  dict_to_bullets(elements, indent=4)
    
    partial_model = f"""Elements: \n {elems_block}"""

    prompt = f"""
You are a modeling assistant. Complete a partial {dsl} for the scenario.

Scenario: {scenario_name}

Ontology annotations (to explain the semantics of each concept element better ):
{ontology_annotation}

Metamodel (allowed element types, relations — follow strictly):
{metamodel_spec}

Current partial model:
{partial_model}

Task:
Propose up to {n_additions} additions to extend the model. An addition could be:
- type: ARCH of the concepts of the metamodel  or CONN that is represented as an edge , or CHAR like a property or constraint.

Constraints:
- Only propose additions that are valid according to the metamodel.
- Do not repeat elements that already exist in the partial model.

Write as a bullet list. Be concise. 
""".strip()


    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )

    output_text = resp.choices[0].message.content
    return prompt, output_text






In [73]:
#iteration_id += 1

prompt, output_text = generate_completion(
    scenario_name=scenario_name,
    elements = {
    "arch": [
        {"process": ["maintenance forecasting", 'maintenance scheduling', "aircraft_routing_optimization", 'spare time allocation', 
                     "slot reservation", 'maintenance execution']},
        {"resource": ["spare parts", "technician"]},
        {"Metric": ["Maintenance Cost", "Parts Availability rate", "aircraft downtime"]},
        { "Entity": ["distribution center","airport"]},

    ],
   

  
    
    
    "connections": [
  
            {"information_flow":["maintenance_forecasting - maintenance_scheduling","aircraft_routing_optimization", "maintenance_execution"]},

            {"uses": ["maintenance execution -spare parts"]},
            {"performs": ["airport - aircraft_routing_optimization"]},
            {"isUsed": ["spare parts - maintenance execution"]},
            {"measures": ["maintenance execution - Maintenance Cost", " maintenance execution - Aircraft Downtime"]},
            {"evaluates_": ["spare parts allocation - Parts Availability rate", 
                            "spare parts allocation - maintenance_execution", 
                            "aircraft_routing_optimization - Aircraft Downtime",
                            "maintenance_execution - Parts Availability rate"]},
           
            {"evaluates": ["maintenance_execution", "Parts Availability rate"]}



    ],
   
    "characteristics": [
         {"Constraint": "maintenance_window"}
      
         
    ]
        


}

,
    model=MODEL,
    temperature=TEMPERATURE,
    n_additions=3,
    dsl="SCM dsl",
    ontology_annotation=ontology_annotation,
    metamodel_spec=metamodel_spec,
)

print("===== PROMPT =====")
print(prompt)
print("\n===== MODEL OUTPUT =====")
print(output_text)


===== PROMPT =====
You are a modeling assistant. Complete a partial SCM dsl for the scenario.

Scenario: Aircraft maintenance routing system

Ontology annotations (to explain the semantics of each concept element better ):
SCM ontology mapping (use exactly these):
- Actor -> Entity
- Item -> Resource
- Interaction -> relationship, InformationFlow, MaterialFlow
- Property -> Metric


Metamodel (allowed element types, relations — follow strictly):
SCM metamodel:

Types  and their fields:

- Flow:
  attributes: name, type, source, destination, frequency, volume(NUMBER)

- Resource:
  attributes: name, type, quantity(NUMBER), cost(NUMBER)
  relations:
    - isUsed[*] -> Process

- Process:
  attributes: name, id, duration, output
  relations:
    - measures[*] -> Metric
    - uses[*] -> Resource
    - produces[*] -> Resource (containment)
    - involves[*] -> Relationship
    - have[*] -> Constraint

- Entity:
  attributes: name, id, location, role
  relations:
    - measures[*] -> Metric 

In [ ]:
# ===== Manually fill this dict (we copy/paste items from output) =====
classified = {
    "arch": [
           
    ],
    "connections": [
       

       {"information_flow": ["maintenance scheduling - maintenance execution"]},
       {"information_flow": ["spare time allocation - slot reservation"]},
    
       
      

    ],
    "characteristics": [
     {"Constraint": "spare_parts_availability"}
    ],
}


# ===== Log everything (prompt + output + youand the  manual dict we accepted to integrate) =====
with txt_path.open("a", encoding="utf-8") as f:
    f.write(f"\n\nITERATION {iteration_id}\n")
    f.write("-" * 80 + "\n")



    f.write("PROMPT\n")
    f.write("-" * 80 + "\n")
    f.write(prompt + "\n\n")

    f.write("MODEL OUTPUT\n")
    f.write("-" * 80 + "\n")
    f.write(output_text.strip() + "\n\n")

    f.write("MANUAL CLASSIFICATION\n")
    f.write("-" * 80 + "\n")
    f.write("ARCH:\n")
    for x in classified["arch"]:
        f.write(f"- {x}\n")
    f.write("\nCONNECTIONS:\n")
    for x in classified["connections"]:
        f.write(f"- {x}\n")
    f.write("\nCHARACTERISTICS:\n")
    for x in classified["characteristics"]:
        f.write(f"- {x}\n")

print(f"\nLogged iteration {iteration_id} to:", txt_path)
classified


Logged iteration 19 to: logs/aircraft_maintenance_routing_system__c_full__20260218-150405.txt


{'arch': [],
 'connections': [{'applies_to': ['spare_parts_availability',
    'maintenance_execution']},
  {'uses': ['aircraft_routing_optimization', 'technician']}],
 'characteristics': [{'Constraint': 'spare_parts_availability'}]}

In [68]:
# ===== Manually fill accepted items for this iteration =====
accepted = {
    "arch": [
            
    ],
    "connections": [





    ],
    "characteristics": [
    ],
}

# ===== Append accepted to the same log file =====
with txt_path.open("a", encoding="utf-8") as f:
    f.write("\n\nACCEPTED (ITERATION {})\n".format(iteration_id))
    f.write("-" * 80 + "\n")

    f.write("ARCH:\n")
    for x in accepted["arch"]:
        f.write(f"- {x}\n")

    f.write("\nCONNECTIONS:\n")
    for x in accepted["connections"]:
        f.write(f"- {x}\n")

    f.write("\nCHARACTERISTICS:\n")
    for x in accepted["characteristics"]:
        f.write(f"- {x}\n")

print(f"Accepted suggestions appended for iteration {iteration_id} to:", txt_path)
accepted


Accepted suggestions appended for iteration 19 to: logs/aircraft_maintenance_routing_system__c_full__20260218-150405.txt


{'arch': [], 'connections': [], 'characteristics': []}

In [69]:




added_elements ={
    "arch": [
        {"Process": "shipment planning"}
    ],
    "connections": [    


    ],
    "characteristics": [
    ],
}
# ===== Append new added line  to the same log file =====
with txt_path.open("a", encoding="utf-8") as f:
    f.write("\n\nADDED ELEMENTS MANUALLY (ITERATION {})\n".format(iteration_id))
    f.write("-" * 80 + "\n")

    f.write("ARCH:\n")
    for x in added_elements["arch"]:
        f.write(f"- {x}\n")

    f.write("\nCONNECTIONS:\n")
    for x in added_elements["connections"]:
        f.write(f"- {x}\n")

    f.write("\nCHARACTERISTICS:\n")
    for x in added_elements["characteristics"]:
        f.write(f"- {x}\n")
   